In [14]:

import hdf5plugin
import h5py
import numpy as np

for filename in ["train.h5", "val.h5"]:
    print(f"\n{'=' * 50}\n{filename}")

    with h5py.File(filename, "r") as f:
        d = f["table/table"]

        print("Rows:", len(d))
        print("Fields:", d.dtype.names)

        # Read only five records
        sample = d[:5]

        for field in d.dtype.names:
            values = sample[field]
            print(f"\n{field}")
            print("Shape:", values.shape)
            print("Dtype:", values.dtype)
            print("Sample:", values)

        # Inspect integer values on a small subset
        ints = d[:10000]["values_block_1"]
        print("\nInteger field ranges:")
        for i in range(ints.shape[1]):
            col = ints[:, i]
            print(
                f"Column {i}: min={col.min()}, "
                f"max={col.max()}, "
                f"unique sample={np.unique(col)[:20]}"
            )



train.h5
Rows: 1211000
Fields: ('index', 'values_block_0', 'values_block_1')

index
Shape: (5,)
Dtype: int64
Sample: [375 377 378 379 380]

values_block_0
Shape: (5, 804)
Dtype: float32
Sample: [[ 474.07114  -250.34703  -223.65196  ...    0.          0.
     0.      ]
 [ 150.50453   120.06239    76.852005 ...    0.          0.
     0.      ]
 [ 251.64539    10.427651 -147.57375  ...    0.          0.
     0.      ]
 [ 451.56613   129.88544   -99.06629  ...    0.          0.
     0.      ]
 [ 399.0939   -168.43208   -47.205597 ...    0.          0.
     0.      ]]

values_block_1
Shape: (5, 2)
Dtype: int64
Sample: [[0 0]
 [0 0]
 [0 0]
 [0 0]
 [0 0]]

Integer field ranges:
Column 0: min=0, max=0, unique sample=[0]
Column 1: min=0, max=1, unique sample=[0 1]

val.h5
Rows: 403000
Fields: ('index', 'values_block_0', 'values_block_1')

index
Shape: (5,)
Dtype: int64
Sample: [954 957 959 970 979]

values_block_0
Shape: (5, 804)
Dtype: float32
Sample: [[ 266.67688    85.2275   -227.71448  ...

In [12]:
pip install hdf5plugin tables pandas

  Using cached hdf5plugin-7.1.0-py3-none-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.2 kB)
  Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached hdf5plugin-7.1.0-py3-none-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (46.4 MB)
Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [hdf5plugin] 1/2 [hdf5plugin]
Note: you may need to restart the kernel to use updated packages.


In [13]:

# Import first to register additional HDF5 filters
import hdf5plugin
import pandas as pd
import numpy as np

for filename in ["train.h5", "val.h5"]:
    print(f"\n{'=' * 50}\n{filename}\n")

    try:
        with pd.HDFStore(filename, mode="r") as store:
            print("Keys:", store.keys())
            df = store.select("table", start=0, stop=5)

        print("Columns:", df.columns.tolist())
        print("Shape:", df.shape)
        print("\nFirst five rows:")
        print(df.to_string())

        print("\nData types:")
        print(df.dtypes.to_string())

    except Exception as e:
        print("Error:", repr(e))



train.h5

Keys: ['/', '/table']
Error: TypeError("a bytes-like object is required, not 'str'")

val.h5

Keys: ['/', '/table']
Error: TypeError("a bytes-like object is required, not 'str'")


In [15]:

import hdf5plugin  # Register extra HDF5 filters
import h5py
import numpy as np

FILES = ["train.h5", "val.h5"]
CHUNK_SIZE = 20_000

for filename in FILES:
    print("\n" + "=" * 65)
    print(filename)

    with h5py.File(filename, "r") as f:
        print("\nRoot keys:", list(f.keys()))
        print("Root attributes:", dict(f.attrs))

        d = f["table/table"]
        print("\nDataset shape:", d.shape)
        print("Dataset dtype:", d.dtype)
        print("Dataset chunks:", d.chunks)
        print("Compression:", d.compression)
        print("Compression options:", d.compression_opts)

        print("\nDataset attributes:")
        for key in d.attrs.keys():
            value = d.attrs[key]
            print(f"{key}: {value}")

        # Check the two integer columns over the entire dataset.
        counts = {}
        feature_min = np.full(804, np.inf)
        feature_max = np.full(804, -np.inf)
        feature_nan = np.zeros(804, dtype=np.int64)
        feature_zero = np.zeros(804, dtype=np.int64)

        for start in range(0, len(d), CHUNK_SIZE):
            block = d[start:start + CHUNK_SIZE]

            ints = block["values_block_1"]
            for col in range(ints.shape[1]):
                vals, nums = np.unique(ints[:, col], return_counts=True)
                for val, num in zip(vals, nums):
                    counts.setdefault(col, {})
                    counts[col][int(val)] = (
                        counts[col].get(int(val), 0) + int(num)
                    )

            x = block["values_block_0"]
            feature_min = np.minimum(feature_min, np.nanmin(x, axis=0))
            feature_max = np.maximum(feature_max, np.nanmax(x, axis=0))
            feature_nan += np.isnan(x).sum(axis=0)
            feature_zero += (x == 0).sum(axis=0)

        print("\nInteger-column value counts:")
        for col, count_dict in counts.items():
            print(f"Column {col}: {count_dict}")

        print("\nFeature summary:")
        print("NaN values:", int(feature_nan.sum()))
        print("Infinite values:", end=" ")
        # A separate full pass would be needed to count infinities exactly.
        print("not counted")
        print("All-zero features:", int(np.sum(feature_max == 0) &
                                         np.sum(feature_min == 0)))
        print("First 30 feature minima:", feature_min[:30])
        print("First 30 feature maxima:", feature_max[:30])
        print("First 30 feature zero counts:", feature_zero[:30])



train.h5

Root keys: ['table']
Root attributes: {'CLASS': np.bytes_(b'GROUP'), 'PYTABLES_FORMAT_VERSION': np.bytes_(b'2.1'), 'TITLE': np.bytes_(b''), 'VERSION': np.bytes_(b'1.0')}

Dataset shape: (1211000,)
Dataset dtype: [('index', '<i8'), ('values_block_0', '<f4', (804,)), ('values_block_1', '<i8', (2,))]
Dataset chunks: (40,)
Compression: unknown
Compression options: None

Dataset attributes:
CLASS: b'TABLE'
FIELD_0_FILL: 0
FIELD_0_NAME: b'index'
FIELD_1_FILL: 0.0
FIELD_1_NAME: b'values_block_0'
FIELD_2_FILL: 0
FIELD_2_NAME: b'values_block_1'
NROWS: 1211000
TITLE: Empty(dtype=dtype('S1'))
VERSION: b'2.7'
index_kind: b'integer'
values_block_0_dtype: b'float32'
values_block_0_kind: b"(lp1\nS'E_0'\np2\naS'PX_0'\np3\naS'PY_0'\np4\naS'PZ_0'\np5\naS'E_1'\np6\naS'PX_1'\np7\naS'PY_1'\np8\naS'PZ_1'\np9\naS'E_2'\np10\naS'PX_2'\np11\naS'PY_2'\np12\naS'PZ_2'\np13\naS'E_3'\np14\naS'PX_3'\np15\naS'PY_3'\np16\naS'PZ_3'\np17\naS'E_4'\np18\naS'PX_4'\np19\naS'PY_4'\np20\naS'PZ_4'\np21\naS'E_5'\np22\

In [4]:

import os
import random
import hdf5plugin
import h5py
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
)

# ---------------- CONFIGURATION ----------------

TRAIN_FILE = "train.h5"
VAL_FILE = "val.h5"
OUT_DIR = "top_cnn_results"

MAX_TRAIN = 200_000
MAX_VAL = 60_000
BATCH_SIZE = 512
EPOCHS = 5
LEARNING_RATE = 1e-3
SEED = 42

# The four truth-level features and ttv are excluded.
# Label: 1 = top signal, 0 = QCD background.
FEATURES_PER_JET = 200
FEATURES_PER_PARTICLE = 4
MOMENTUM_SCALE = 650.0

os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))

device = torch.device("cpu")
print("Device:", device)


# ---------------- DATA LOADING ----------------

def load_dataset(filename, limit):
    print(f"\nLoading {filename} ...")

    with h5py.File(filename, "r") as f:
        d = f["table/table"]
        n = min(limit, len(d))
        records = d[:n]

    # First 800 values = 200 constituents x (E, PX, PY, PZ).
    # truthE, truthPX, truthPY, truthPZ are deliberately excluded.
    X = records["values_block_0"][:, :800].copy()
    X = X.reshape(n, FEATURES_PER_JET, FEATURES_PER_PARTICLE)

    # values_block_1 = [ttv, is_signal_new]
    y = records["values_block_1"][:, 1].astype(np.int64)

    del records

    # Basic data-integrity checks.
    if not np.isfinite(X).all():
        raise ValueError(f"NaN or infinity found in {filename}")

    if not np.isin(y, [0, 1]).all():
        raise ValueError(f"Unexpected labels in {filename}: {np.unique(y)}")

    # Scale momenta to a more convenient numerical range.
    X /= MOMENTUM_SCALE

    unique, counts = np.unique(y, return_counts=True)
    print("Events:", n)
    print("Label counts:", dict(zip(unique.tolist(), counts.tolist())))
    print("Input shape:", X.shape)
    print("Label mapping: 0 = QCD, 1 = top")

    return X, y


X_train, y_train = load_dataset(TRAIN_FILE, MAX_TRAIN)
X_val, y_val = load_dataset(VAL_FILE, MAX_VAL)

# Shuffle training events reproducibly.
rng = np.random.default_rng(SEED)
permutation = rng.permutation(len(y_train))
X_train = X_train[permutation]
y_train = y_train[permutation]

# PyTorch Conv1d expects [batch, channels, sequence length].
# Keep the stored data as [events, constituents, features] until batching.
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train.astype(np.float32)),
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val.astype(np.float32)),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

del X_train, y_train


# ---------------- CNN MODEL ----------------

class TopTaggingCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(4, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.GELU(),
        )

        # Combine max and average pooling over constituent positions.
        self.pool_max = nn.AdaptiveMaxPool1d(1)
        self.pool_avg = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # [batch, 200, 4] -> [batch, 4, 200]
        x = x.transpose(1, 2)
        x = self.conv(x)

        x_max = self.pool_max(x).squeeze(-1)
        x_avg = self.pool_avg(x).squeeze(-1)

        x = torch.cat([x_max, x_avg], dim=1)
        return self.classifier(x).squeeze(1)


model = TopTaggingCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

print("\nTrainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


# ---------------- TRAINING / EVALUATION ----------------

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * len(yb)
            probs = torch.sigmoid(logits)

            all_labels.append(yb.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels).astype(int)
    probs = np.concatenate(all_probs)
    avg_loss = total_loss / len(loader.dataset)
    auc = roc_auc_score(labels, probs)

    return avg_loss, auc, labels, probs


history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": [],
}

best_auc = -1.0
best_path = os.path.join(OUT_DIR, "best_model.pt")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch_idx, (xb, yb) in enumerate(train_loader, start=1):
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)

        if batch_idx % 50 == 0:
            print(
                f"Epoch {epoch}/{EPOCHS} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = total_loss / len(train_loader.dataset)
    val_loss, val_auc, _, _ = evaluate(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)

    print(
        f"\nEpoch {epoch}/{EPOCHS}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"val_AUC={val_auc:.5f}\n"
    )

    # Keep the checkpoint with the best validation AUC.
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), best_path)

# Reload the best checkpoint before final validation metrics.
model.load_state_dict(torch.load(best_path, map_location=device))
val_loss, val_auc, y_true, y_score = evaluate(val_loader)
y_pred = (y_score >= 0.5).astype(int)

print("\n========== VALIDATION RESULTS ==========")
print(f"Validation loss: {val_loss:.5f}")
print(f"Validation ROC-AUC: {val_auc:.5f}")
print(f"Validation accuracy at threshold 0.5: "
      f"{accuracy_score(y_true, y_pred):.5f}")
print("\nClassification report:")
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["QCD background", "Top signal"],
    digits=4,
    zero_division=0,
)
print(report)

with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)
    f.write(f"\nValidation ROC-AUC: {val_auc:.6f}\n")
    f.write(f"Validation loss: {val_loss:.6f}\n")


# ---------------- PLOTS ----------------

# 1. Training and validation loss.
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], marker="o", label="Training loss")
plt.plot(history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("CNN training history")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_history.png"), dpi=160)
plt.close()

# 2. Validation ROC curve.
fpr, tpr, thresholds = roc_curve(y_true, y_score)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {val_auc:.4f}")
plt.plot([0, 1], [0, 1], "--", label="Random classifier")
plt.xlabel("False positive rate (QCD misclassified as top)")
plt.ylabel("True positive rate (top signal efficiency)")
plt.title("Validation ROC curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_curve.png"), dpi=160)
plt.close()

# 3. Confusion matrix at probability threshold 0.5.
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["QCD", "Top"],
)
disp.plot(values_format="d")
plt.title("Validation confusion matrix (threshold = 0.5)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=160)
plt.close()

# 4. Classifier score distributions.
plt.figure(figsize=(8, 5))
plt.hist(
    y_score[y_true == 0],
    bins=50,
    range=(0, 1),
    density=True,
    histtype="step",
    linewidth=1.8,
    label="QCD background",
)
plt.hist(
    y_score[y_true == 1],
    bins=50,
    range=(0, 1),
    density=True,
    histtype="step",
    linewidth=1.8,
    label="Top signal",
)
plt.xlabel("CNN predicted top probability")
plt.ylabel("Normalised density")
plt.title("Classifier score distributions")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "score_distributions.png"), dpi=160)
plt.close()

# 5. Signal efficiency vs. background rejection.
# Background rejection = 1 / QCD false-positive rate.
valid = fpr > 0
rejection = 1.0 / fpr[valid]

plt.figure(figsize=(8, 5))
plt.plot(tpr[valid], rejection)
plt.yscale("log")
plt.xlabel("Top signal efficiency")
plt.ylabel("QCD background rejection (1 / FPR)")
plt.title("Signal efficiency vs. background rejection")
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "signal_efficiency_vs_rejection.png"),
            dpi=160)
plt.close()

# 6. Validation AUC versus epoch.
plt.figure(figsize=(8, 5))
plt.plot(
    range(1, EPOCHS + 1),
    history["val_auc"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Validation ROC-AUC")
plt.title("Validation AUC history")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "validation_auc.png"), dpi=160)
plt.close()

print("\nSaved outputs in:", OUT_DIR)
print("Best checkpoint:", best_path)
print("Plots and report generated.")
print("These are validation results, not independent test results.")


Device: cpu

Loading train.h5 ...
Events: 200000
Label counts: {0: 100078, 1: 99922}
Input shape: (200000, 200, 4)
Label mapping: 0 = QCD, 1 = top

Loading val.h5 ...
Events: 60000
Label counts: {0: 29440, 1: 30560}
Input shape: (60000, 200, 4)
Label mapping: 0 = QCD, 1 = top

Trainable parameters: 31841
Epoch 1/5 | Batch 50/391 | Loss 0.5730
Epoch 1/5 | Batch 100/391 | Loss 0.5009
Epoch 1/5 | Batch 150/391 | Loss 0.5148
Epoch 1/5 | Batch 200/391 | Loss 0.4550
Epoch 1/5 | Batch 250/391 | Loss 0.4624
Epoch 1/5 | Batch 300/391 | Loss 0.4034
Epoch 1/5 | Batch 350/391 | Loss 0.4063

Epoch 1/5: train_loss=0.4794, val_loss=0.3782, val_AUC=0.90185

Epoch 2/5 | Batch 50/391 | Loss 0.3425
Epoch 2/5 | Batch 100/391 | Loss 0.3752
Epoch 2/5 | Batch 150/391 | Loss 0.3571
Epoch 2/5 | Batch 200/391 | Loss 0.4000
Epoch 2/5 | Batch 250/391 | Loss 0.3295
Epoch 2/5 | Batch 300/391 | Loss 0.3212
Epoch 2/5 | Batch 350/391 | Loss 0.3148

Epoch 2/5: train_loss=0.3448, val_loss=0.6711, val_AUC=0.91159

Epoch 